Google Colab Setup and Execution Guide for Adaptive Graph Construction Experiments

This script sets up the environment and runs the full pipeline on Google Colab.
Run this in Colab cells in order.

Prerequisites: Colab GPU instance (free tier has T4)


In [1]:
import sys; print(sys.version)

3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


## CELL 0: Change Python version (REQUIRED if Colab gives you Python 3.12)

DGL's official wheels only support Python <= 3.11. If `!python --version` below reports 3.12, pick **one** of the options below **before** cloning the repo (a runtime restart wipes `/content`).

**Option A - Switch the Colab runtime (easiest, no code).**
`Runtime` -> `Change runtime type` -> set Python version to 3.11 -> `Save`. In the VS Code Colab extension, click the runtime/kernel name at the top of the notebook and pick the 3.11 runtime. Then re-run from CELL 1.

**Option B - Install a Python 3.11 interpreter for subprocess use only.**
Useful if you just need `!python3.11 script.py` to run something. The notebook kernel itself stays on the default version, so `import dgl` in a notebook cell will still fail.
```
!sudo apt-get install -y python3.11 python3.11-distutils
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.11
!python3.11 -m pip install <packages>
```

**Option C - Swap the whole environment with condacolab (recommended for this notebook).**
This actually changes the kernel Python, so `import dgl` works. It **restarts the runtime**, so run this in a fresh session before CELL 1. The code cell below does this.

In [2]:
!nvidia-smi

Sun May 24 12:23:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
# Option C: install condacolab and downgrade to Python 3.11.
# This RESTARTS the runtime automatically. After the restart, run the next cell
# to confirm the version, then continue with CELL 1 (clone repo).

!pip install -q condacolab
import condacolab
condacolab.install()  # triggers a kernel restart

# After the restart, run these in a new cell to confirm / downgrade further if needed:
# !conda install -y python=3.11
# !python --version

✨🍰✨ Everything looks OK!


In [17]:
!nvidia-smi

Sat May 23 18:33:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P8             24W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## CELL 1: Mount Google Drive and clone repository from GitHub


In [2]:
# Mount Google Drive
# from google.colab import drive
# drive.mount('/gdrive')

import os
import subprocess
from pathlib import Path

os.chdir('/content')

# Clone GeoContrastNet repo from GitHub
# IMPORTANT: Replace URL with your actual GitHub repo
GITHUB_REPO = "https://github.com/NilBiescas/GeoContrastNet"
REPO_DIR = Path("/content/GeoContrastNet")

print("=" * 70)
print("STEP 1: Clone GeoContrastNet from GitHub")
print("=" * 70)
print(f"Repository URL: {GITHUB_REPO}")
print("(If repo is private, use: https://<TOKEN>@github.com/<user>/GeoContrastNet.git)")
print()

if REPO_DIR.exists():
    if (REPO_DIR / ".git").exists():
        print(f"[INFO] {REPO_DIR} already exists. Updating it...")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        raise RuntimeError(
            f"{REPO_DIR} already exists but is not a git repository. "
            "Rename or remove it, then rerun this cell."
        )
else:
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)

if not (REPO_DIR / ".git").exists():
    raise RuntimeError(f"Clone failed: {REPO_DIR} was not created as a git repository.")

os.chdir(REPO_DIR)
print(f"\n[SUCCESS] Repository ready at {REPO_DIR}")
print("\nRepository contents:")
for name in sorted(os.listdir(REPO_DIR))[:20]:
    print(" ", name)


STEP 1: Clone GeoContrastNet from GitHub
Repository URL: https://github.com/NilBiescas/GeoContrastNet
(If repo is private, use: https://<TOKEN>@github.com/<user>/GeoContrastNet.git)


[SUCCESS] Repository ready at /content/GeoContrastNet

Repository contents:
  .git
  .gitignore
  README.md
  V2_contrastive_datasets.py
  build_graphs.py
  graphs_stage1
  graphs_stage2
  main.py
  paths.py
  pau_eval.py
  setups_stage1
  setups_stage2
  src
  utils.py


## CELL 2: Install dependencies


In [3]:
!python --version

Python 3.11.11


In [4]:
# IMPORTANT: skip the condacolab cell above. It leaves you with a split env
# where %pip writes to Python 3.11 but the kernel runs Python 3.12, so installs
# never reach the kernel. Python 3.12 + DGL 2.4 works fine with the wheel index below.

import sys, subprocess

# Anchor every install to the KERNEL'S OWN python — don't trust `%pip` or `!pip`.
PY = sys.executable
print(f"Installing into kernel's interpreter: {PY}")
print(f"Kernel Python version: {sys.version.split()[0]}\n")

def pip(*args):
    subprocess.run([PY, "-m", "pip", *args], check=True)

# 1. Uninstall stale torch/dgl/torchdata to avoid any "Requirement already satisfied" no-ops.
pip("uninstall", "-y", "torch", "torchvision", "torchaudio", "torchdata", "dgl")

# 2. Install non-torch deps FIRST. spacy + co don't need torch — installing them now
#    means later torch-pinning won't be disturbed by their dep resolution.
pip("install", "scikit-learn", "pillow", "pyyaml", "matplotlib", "tqdm",
    "requests", "pandas", "seaborn", "spacy")
subprocess.run([PY, "-m", "spacy", "download", "en_core_web_lg"], check=True)

# 3. Pin PyTorch to 2.4.0 (DGL ships matching libgraphbolt_pytorch_2.4.0.so).
pip("install", "torch==2.4.0", "torchvision==0.19.0", "torchaudio==2.4.0",
    "--index-url", "https://download.pytorch.org/whl/cu121")

# 4. Install DGL with --no-deps so pip can't re-upgrade torch behind our back.
pip("install", "--no-deps", "dgl==2.4.0",
    "-f", "https://data.dgl.ai/wheels/torch-2.4/cu121/repo.html")

# 5. torchdata last, also --no-deps (its torch constraint would otherwise drag in latest).
pip("install", "--no-deps", "torchdata==0.7.1")

# 6. Sanity-check what landed on disk (no import of torch yet — that needs a kernel restart).
print("\n=== Installed versions ===")
for pkg in ("torch", "dgl", "torchdata"):
    r = subprocess.run([PY, "-m", "pip", "show", pkg], capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if line.startswith(("Name:", "Version:", "Location:")):
            print(f"  {line}")
    print()

print("=" * 70)
print("INSTALL COMPLETE. Now RESTART THE KERNEL before running the next cell.")
print("  VS Code: click kernel name at top of notebook -> Restart.")
print("=" * 70)


Installing into kernel's interpreter: /usr/bin/python3.real
Kernel Python version: 3.11.13


=== Installed versions ===
  Name: torch
  Version: 2.4.0+cu121
  Location: /usr/local/lib/python3.11/dist-packages

  Name: dgl
  Version: 2.4.0+cu121
  Location: /usr/local/lib/python3.11/dist-packages

  Name: torchdata
  Version: 0.7.1
  Location: /usr/local/lib/python3.11/dist-packages

INSTALL COMPLETE. Now RESTART THE KERNEL before running the next cell.
  VS Code: click kernel name at top of notebook -> Restart.


In [5]:
# Run this AFTER restarting the kernel. If you still see PyTorch 2.10 here,
# the restart did not take effect — restart again from the kernel picker.
import sys, torch, dgl
print(f'Kernel Python: {sys.version.split()[0]}')
print(f'PyTorch:       {torch.__version__}   CUDA available: {torch.cuda.is_available()}')
print(f'DGL:           {dgl.__version__}')

# Sanity: build a tiny graph on GPU to confirm the full stack works
g = dgl.graph(([0, 1, 2], [1, 2, 0]))
if torch.cuda.is_available():
    g = g.to('cuda')
    print(f'Test graph on device: {g.device}')
print('OK — DGL is functional.')


DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
Kernel Python: 3.11.13
PyTorch:       2.4.0+cu121   CUDA available: True
DGL:           2.4.0+cu121
Test graph on device: cuda:0
OK — DGL is functional.


In [6]:
# DIAGNOSTIC — run this to see what's actually installed and which graphbolt .so files exist.
# No imports of torch/dgl, so it won't fail.
import subprocess, sys, os, glob

print("=== pip show torch ===")
print(subprocess.run([sys.executable, "-m", "pip", "show", "torch"], capture_output=True, text=True).stdout)

print("=== pip show dgl ===")
print(subprocess.run([sys.executable, "-m", "pip", "show", "dgl"], capture_output=True, text=True).stdout)

print("=== pip show torchdata ===")
print(subprocess.run([sys.executable, "-m", "pip", "show", "torchdata"], capture_output=True, text=True).stdout)

# Find the dgl install directory and list its graphbolt .so files
import importlib.util
spec = importlib.util.find_spec("dgl")
if spec and spec.origin:
    dgl_dir = os.path.dirname(spec.origin)
    print(f"=== DGL install dir: {dgl_dir} ===")
    so_files = sorted(glob.glob(os.path.join(dgl_dir, "graphbolt", "*.so")))
    if so_files:
        print("graphbolt .so files present:")
        for f in so_files:
            print(f"  {os.path.basename(f)}")
    else:
        print("No .so files in graphbolt/ subdir.")
else:
    print("dgl package not found.")


=== pip show torch ===
Name: torch
Version: 2.4.0+cu121
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3
Location: /usr/local/lib/python3.11/dist-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-nccl-cu12, nvidia-nvtx-cu12, sympy, triton, typing-extensions
Required-by: accelerate, dgl, fastai, peft, sentence-transformers, timm, torchaudio, torchdata, torchvision

=== pip show dgl ===
Name: dgl
Version: 2.4.0+cu121
Summary: Deep Graph Library
Home-page: https://github.com/dmlc/dgl
Author: 
Author-email: 
License: APACHE
Location: /usr/local/lib/python3.11/dist-packages
Requires: networkx, numpy, packaging, pandas, psutil, pydantic, pyyaml, reques

## CELL 3: Download FUNSD dataset (if not already present)


In [7]:
import os
data_dir = '/content/GeoContrastNet/src/data/datasets/FUNSD'
if not os.path.exists(data_dir):
    print("FUNSD not found, downloading...")
    os.chdir('/content/GeoContrastNet/src/data')
    !python download.py
    os.chdir('/content/GeoContrastNet')
    print("FUNSD download complete!")
else:
    print(f"FUNSD already exists at {data_dir}")

!ls -la src/data/datasets/FUNSD/ | head -20


FUNSD not found, downloading...
Traceback (most recent call last):
  File "/content/GeoContrastNet/src/data/download.py", line 5, in <module>
    import wget
ModuleNotFoundError: No module named 'wget'
FUNSD download complete!
ls: cannot access 'src/data/datasets/FUNSD/': No such file or directory


## CELL 4: Build both graph variants (Stage 1)


In [8]:
import subprocess
import sys

print("\n" + "="*70)
print("STAGE 1: Building KNN and Adaptive graphs")
print("="*70 + "\n")

result = subprocess.run([sys.executable, 'scripts/build_baseline_vs_adaptive.py'],
                       cwd='/content/GeoContrastNet')

if result.returncode == 0:
    print("\n[SUCCESS] Graph building complete!")
    !ls -la graphs_stage1/adaptive_experiment/
else:
    print("\n[ERROR] Graph building failed!")
    sys.exit(1)



STAGE 1: Building KNN and Adaptive graphs


[ERROR] Graph building failed!


SystemExit: 1

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## CELL 5: Visualize graphs (optional, helps debug)


In [ ]:
print("\n" + "="*70)
print("Visualizing KNN vs Adaptive graphs")
print("="*70 + "\n")

result = subprocess.run([sys.executable, 'scripts/visualize_graphs.py'],
                       cwd='/content/GeoContrastNet')

print("\n[SUCCESS] Visualizations saved to outputs/graph_compare/")
!ls -la outputs/graph_compare/ | head -20


## CELL 6: Train Stage 2 on KNN baseline


In [ ]:
print("\n" + "="*70)
print("STAGE 2a: Training on KNN baseline graphs (100 epochs)")
print("Estimated time: ~20-30 min on GPU, ~2-4 hours on CPU")
print("="*70 + "\n")

result = subprocess.run([sys.executable, 'main.py', '--run-name', 'run111_knn'],
                       cwd='/content/GeoContrastNet')

if result.returncode == 0:
    print("\n[SUCCESS] KNN training complete!")
    !ls -la runs/run111_knn/
else:
    print("\n[ERROR] KNN training failed!")


## CELL 7: Train Stage 2 on adaptive graphs


In [ ]:
print("\n" + "="*70)
print("STAGE 2b: Training on adaptive graphs (100 epochs)")
print("Estimated time: ~20-30 min on GPU, ~2-4 hours on CPU")
print("="*70 + "\n")

result = subprocess.run([sys.executable, 'main.py', '--run-name', 'run111_adaptive'],
                       cwd='/content/GeoContrastNet')

if result.returncode == 0:
    print("\n[SUCCESS] Adaptive training complete!")
    !ls -la runs/run111_adaptive/
else:
    print("\n[ERROR] Adaptive training failed!")


## CELL 8: Collect and compare results


In [ ]:
print("\n" + "="*70)
print("Comparing KNN vs Adaptive results")
print("="*70 + "\n")

result = subprocess.run([sys.executable, 'scripts/collect_results.py'],
                       cwd='/content/GeoContrastNet')

print("\n✓ Results aggregated!")

# Display comparison
import pandas as pd
from pathlib import Path

comparison_csv = Path('/content/GeoContrastNet/outputs/comparison.csv')
if comparison_csv.exists():
    df = pd.read_csv(comparison_csv)
    print("\n" + "="*70)
    print("COMPARISON RESULTS")
    print("="*70)
    print(df.to_string())


## CELL 9: Save results back to Google Drive (thesis2 folder)


In [ ]:
print("\n" + "="*70)
print("Saving results to Google Drive")
print("="*70)

import shutil
import os

# Ensure thesis2 folder exists
!mkdir -p /gdrive/'My Drive'/thesis2

# Copy the full GeoContrastNet folder with results back to Drive
print("\nCopying runs/ and outputs/ to Google Drive...")
!cp -r /content/GeoContrastNet/runs "/gdrive/My Drive/thesis2/" 2>/dev/null && echo "OK: Copied runs/" || echo "WARN: Could not copy runs"
!cp -r /content/GeoContrastNet/outputs "/gdrive/My Drive/thesis2/" 2>/dev/null && echo "OK: Copied outputs/" || echo "WARN: Could not copy outputs"
!cp -r /content/GeoContrastNet/graphs_stage1 "/gdrive/My Drive/thesis2/" 2>/dev/null && echo "OK: Copied graphs_stage1/" || echo "WARN: Could not copy graphs"

# Also create a zip for easy download
print("\nCreating backup zip file...")
!cd /content/GeoContrastNet && zip -r "/gdrive/My Drive/thesis2/GeoContrastNet_results_backup.zip" runs outputs graphs_stage1 2>/dev/null

print("\nAll results saved to /gdrive/My Drive/thesis2/")
print("Files available:")
!ls -lh "/gdrive/My Drive/thesis2/" | head -20

# Final Summary

print("\n" + "="*70)
print("PIPELINE COMPLETE!")
print("="*70)
print("\nAll results saved to: /gdrive/My Drive/thesis2/")
print("\nFiles generated:")
print("  - runs/run111_knn/ (baseline results)")
print("  - runs/run111_adaptive/ (adaptive results)")
print("  - outputs/comparison.csv (metrics summary)")
print("  - outputs/comparison.json (detailed metrics)")
print("  - outputs/graph_compare/*.png (visualizations)")
print("  - graphs_stage1/adaptive_experiment/ (graph binaries)")
print("\nNext steps:")
print("  1. Download results from Google Drive")
print("  2. Review comparison.csv for metrics")
print("  3. Check visualizations in graph_compare/")
print("  4. Document findings for your thesis")
print("\nFor full documentation, see: ADAPTIVE_GRAPH_CONSTRUCTION.md")
